## One request, start to finish

**What happens between a customer typing a question and the answer appearing. Each box is code you write in the Part named in it.**

```mermaid
flowchart TD
    C(["a customer<br/>on the shop's website"]) -->|"POST /v1/chat<br/>question + token"| I{{"identify() · P3<br/>which customer is this?"}}
    I --> T["find_thread() · P2<br/>which conversation?"]
    T --> A["the agent · P1<br/>build_agent(customer)"]
    A <-->|"messages"| M["the model · P0<br/>gpt-4.1-mini on OpenAI<br/>or gpt-oss:20b in Ollama"]
    A <-->|"load · save"| S[("conversation store · P4<br/>the checkpointer")]
    A --> TL["three tools · P1<br/>look_up_order · price_order<br/>start_return"]
    TL -->|"all three read"| DB[("online_retail.db<br/>the shop's orders, read-only")]
    TL -->|"only start_return writes"| L[("the ledger · P1, P6<br/>return requests · refunds")]
    A --> R(["Answer · P1<br/>reply · tools used · tokens"])

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef tools fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef guard fill:#fce8e6,stroke:#d93025,color:#a50e0e
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333
    class A,M model
    class T,TL tools
    class S,DB,L stored
    class I guard
    class C,R endpoint
```

- **`identify()`** reads the sign-in token and decides which customer is asking (P3).
- **`find_thread()`** finds the conversation to continue, or starts a new one. Conversations are kept in the **checkpointer**, which P4 moves out of the process so that a restart doesn't lose them.
- **The agent** is the model plus three tools, built fresh for this customer, so the tools can only see this customer's orders (P1).
- **The model** runs wherever one setting, `PORTER_PROVIDER`, says: gpt-4.1-mini on OpenAI's API (the default), or gpt-oss:20b in Ollama. The agent talks to both the same way (P0).
- **All three tools read the shop's orders, and never change them. `start_return` is the only tool that writes anything,** and it writes to the **ledger**: the service's own record of return requests and refunds (P1). Paying a refund is a separate step, for a member of staff (P6).
- The route sends back an **`Answer`**: the reply, the tools that were used, and the tokens the run cost (P1, P2).

# Part 0

### P0 in one picture

**One setting chooses where the model runs. The notebook agent is the same either way, and it answers correctly: for one person, in this kernel.**

```mermaid
---
config:
  flowchart:
    wrappingWidth: 360
---
flowchart TD
    P[".env · PORTER_PROVIDER"] -->|"openai, the default"| OAI["gpt-4.1-mini at api.openai.com<br/>key: OPENAI_API_KEY"]
    P -->|"ollama"| OLL["gpt-oss:20b at 127.0.0.1:11434<br/>no key"]
    OAI --> CL["ChatOpenAI(model, base_url, api_key)<br/>one client: both serve Chat Completions"]
    OLL --> CL
    Q(["“When did I place order 580638?”"]) --> A["notebook_agent<br/>create_agent: the model + look_up_order"]
    CL --> A
    A -->|"1 · the model asks<br/>for a tool"| T["look_up_order(“580638”)"]
    T <--> DB[("online_retail.db")]
    T -->|"2 · “Order 580638:<br/>placed 2011-12-05<br/>12:44:00, not<br/>cancelled.”"| A
    A -->|"3 · no more tools:<br/>the model replies"| R(["the reply: placed on 5 December 2011, at 12:44"])
    R --> NINE{{"it works, for one person, in this kernel<br/>nine things break once anybody else uses it"}}

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef tools fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef guard fill:#fce8e6,stroke:#d93025,color:#a50e0e
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333
    class A,OAI,OLL,CL model
    class T tools
    class P,DB stored
    class NINE guard
    class Q,R endpoint
```

- **The loop is what `create_agent` builds:** the model asks for a tool, the tool runs, its result goes back to the model, and the model writes the reply once it needs nothing more.
- **The provider changes one box.** OpenAI and Ollama both serve the Chat Completions API, so the same `ChatOpenAI` client talks to either.
- The red box at the bottom is the table at the end of P0: the nine things the rest of the notebook fixes.

In [11]:
# From here on, each cell prints how long it took.
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 236 µs (started: 2026-09-29 21:45:47 +05:30)


In [12]:



from dotenv import load_dotenv

load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")


True

time: 1.24 ms (started: 2026-09-29 21:45:47 +05:30)


In [13]:
# The imports the rest of the notebook uses, the paths, and the model.
import asyncio, json, os, shutil, sqlite3, subprocess, time, uuid
from pathlib import Path

import httpx                        # an HTTP client: talks to the model endpoint and, in P7–P8, to the running service
from dotenv import load_dotenv      # reads the NAME=value lines of .env into environment variables

ROOT = Path.cwd()                   # this folder; every path below is built from it
DB = ROOT / "online_retail.db"      # the shop's orders
RESULTS = ROOT / "results"          # saved runs of the P7 and P8 checks (see RUN_LIVE, below)

# The model, and later the signing secret, come from .env (the table above). The provider fills in
# whatever .env leaves out; Settings, in P1, makes the same decision.
load_dotenv(ROOT / ".env")
PROVIDER = os.getenv("PORTER_PROVIDER", "openai")                  # "openai" or "ollama"
DEFAULT_MODEL, DEFAULT_URL = {"openai": ("gpt-4.1-mini", "https://api.openai.com/v1"),
                              "ollama": ("gpt-oss:20b", "http://127.0.0.1:11434/v1")}[PROVIDER]
MODEL = os.getenv("PORTER_MODEL") or DEFAULT_MODEL
BASE_URL = os.getenv("PORTER_BASE_URL") or DEFAULT_URL
# The OpenAI key is sent to OpenAI and nowhere else. Ollama checks no key; a tunnel checks its own token.
if os.getenv("PORTER_API_KEY"):
    API_KEY = os.getenv("PORTER_API_KEY")
elif BASE_URL.startswith("https://api.openai.com"):
    API_KEY = os.getenv("OPENAI_API_KEY", "")
else:
    API_KEY = "ollama"

# A scratch folder for everything this notebook writes: conversations, ledgers. Emptied here,
# so every run starts from nothing and every count below is this run's.
SCRATCH = ROOT / ".scratch"
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
# The service reads these two variables to decide where its own files go (Settings, in P1).
os.environ["PORTER_CHECKPOINT_PATH"] = str(SCRATCH / "threads.sqlite")
os.environ["PORTER_LEDGER_PATH"] = str(SCRATCH / "ledger.sqlite")

# Runs that need infrastructure (a container, the tunnel, the AWS instance) save to results/.
# False loads the saved run; True runs it again.
RUN_LIVE = False

print("model    ", MODEL, "at", BASE_URL, f"(provider: {PROVIDER})")
print("database ", DB.name, f"{DB.stat().st_size / 1e6:.0f} MB")

model     gpt-4.1-mini at https://api.openai.com/v1 (provider: openai)
database  online_retail.db 35 MB
time: 2.52 ms (started: 2026-09-29 21:45:47 +05:30)


In [14]:
# Each table's size, its columns, and its first three rows.
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)      # mode=ro: opened read-only, so SQLite refuses any write
for table in ("invoices", "line_items", "products"):
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    columns = [c[1] for c in con.execute(f"PRAGMA table_info({table})")]    # one row per column; c[1] is its name
    print(f"{table}  ({count:,} rows)  {' · '.join(columns)}")
    for row in con.execute(f"SELECT * FROM {table} WHERE rowid IN (1, 2, 3)"):
        print("   ", row)
con.close()

invoices  (25,900 rows)  invoice_no · customer_id · invoice_ts · country · is_cancelled
    ('536365', 17850.0, '2010-12-01 08:26:00', 'United Kingdom', 0)
    ('536366', 17850.0, '2010-12-01 08:28:00', 'United Kingdom', 0)
    ('536367', 13047.0, '2010-12-01 08:34:00', 'United Kingdom', 0)
line_items  (541,909 rows)  invoice_no · stock_code · quantity · unit_price
    ('536365', '85123A', 6, 2.55)
    ('536365', '71053', 6, 3.39)
    ('536365', '84406B', 8, 2.75)
products  (3,958 rows)  stock_code · description
    ('10002', 'INFLATABLE POLITICAL GLOBE ')
    ('10080', 'GROOVY CACTUS INFLATABLE')
    ('10120', 'DOGGY RUBBER')
time: 3.14 ms (started: 2026-09-29 21:45:47 +05:30)


In [15]:
# Porter as a first notebook version: one tool, one agent, one question. It works.
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

CUSTOMER = 12381.0          # the customer this notebook follows (ids are decimals in this data)

# @tool turns a plain function into a tool: its name, arguments and docstring become the
# description the model reads when it decides what to call.
@tool
def look_up_order(invoice_no: str) -> str:
    """Look up an order: when it was placed and whether it was cancelled."""
    con = sqlite3.connect("online_retail.db")            # relative path, read-write
    row = con.execute("SELECT invoice_no, invoice_ts, is_cancelled FROM invoices WHERE invoice_no = ?",
                      (invoice_no,)).fetchone()          # no customer anywhere in the query
    con.close()
    return f"Order {row[0]}: placed {row[1]}, {'cancelled' if row[2] else 'not cancelled'}." if row else "No such order."

# create_agent runs a loop: the model asks for a tool, the tool runs, its result goes back to
# the model, and so on until the model writes a reply instead of asking for a tool.
notebook_agent = create_agent(
    model=ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY, temperature=0),
    tools=[look_up_order],
    system_prompt="You are Porter, the order desk for Wickmere & Rook. Answer using the tool.",
    checkpointer=InMemorySaver(),       # keeps each conversation in this Python process's memory
)
# One question. The thread_id names the conversation: asking again with "demo" would continue it.
out = notebook_agent.invoke({"messages": [{"role": "user", "content": "When did I place order 580638?"}]},
                            {"configurable": {"thread_id": "demo"}})
print(out["messages"][-1].content)      # the last message is the model's reply

You placed order 580638 on December 5, 2011 at 12:44 PM.
time: 4.43 s (started: 2026-09-29 21:45:47 +05:30)


# Part 1

### P1 in one picture

**Everything a request needs becomes an argument or a setting, so a process that has never seen this kernel builds the same agent.**

```mermaid
---
config:
  flowchart:
    wrappingWidth: 360
---
flowchart TD
    X{{"a web server starts fresh, from files<br/>notebook_agent, MODEL, CUSTOMER: not there"}} -.->|"the fix: every input<br/>is an argument<br/>or a setting"| S
    S["Settings()<br/>read from .env and PORTER_* variables<br/>typed, and checked at start-up"] -.->|"a 90 s model call<br/>in a 60 s request"| NO{{"refused at startup"}}
    Q(["answer(“How much was order 580638?”, customer_id=12381)"]) --> BA
    S --> BA["build_agent(12381, settings, checkpointer)<br/>limits: 6 model calls · 8 tool calls<br/>and a retry on a passing failure"]
    BA --> BM["build_model(settings)<br/>gpt-4.1-mini, with a time limit"]
    BA --> BT["build_tools(12381, …)<br/>look_up_order · price_order · start_return<br/>every query: AND customer_id = 12381"]
    BT -->|"read-only"| DB[("online_retail.db")]
    BT -->|"start_return only"| L[("the ledger · returns")]
    BT -.->|"536365, another<br/>customer's order"| NF{{"“No order '536365' on this account.”"}}
    BM --> RUN["agent.invoke() → _collect()"]
    BT --> RUN
    RUN --> R(["Answer<br/>reply: the total, 147.01 · tools_used: price_order<br/>usage: 2 model calls, the tokens, the seconds"])

    classDef model fill:#e8f0fe,stroke:#4285f4,color:#174ea6
    classDef tools fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef stored fill:#f3e8fd,stroke:#9334e6,color:#681da8
    classDef guard fill:#fce8e6,stroke:#d93025,color:#a50e0e
    classDef endpoint fill:#e6f4ea,stroke:#34a853,color:#137333
    class BM,RUN model
    class S,BA,BT tools
    class DB,L stored
    class X,NO,NF guard
    class Q,R endpoint
```

- `Settings` is read once and checked. `build_agent()` puts the model, this customer's tools and the limits together for one request, and `answer()` returns a typed `Answer`.
- **The dotted paths are refusals:** a configuration that can't work never starts, and another customer's order gets the same reply as one that doesn't exist.

In [16]:
# Settings: every configurable value, read from the environment and checked at start-up.
from typing import Literal

from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

# The two places the model can run, and what each one fills in (P0's table).
PROVIDERS = {
    "openai": {"model": "gpt-4.1-mini", "base_url": "https://api.openai.com/v1"},
    "ollama": {"model": "gpt-oss:20b", "base_url": "http://127.0.0.1:11434/v1"},
}

# Every setting the service has, in one class. Each field is read from the environment variable
# PORTER_<FIELD>: model from PORTER_MODEL, max_model_calls from PORTER_MAX_MODEL_CALLS, and so on.
# The value is converted to the type written after the name; one that can't be converted stops
# the start, naming the field.
class Settings(BaseSettings):
    # env_prefix: the PORTER_ in front of every name. env_file: read .env as well.
    # extra="ignore": other lines in .env (NGROK_DOMAIN, …) are not an error.
    model_config = SettingsConfigDict(env_prefix="PORTER_", env_file=ROOT / ".env", extra="ignore")

    # the model: the provider fills in model, base_url and api_key, unless they are set
    provider: Literal["openai", "ollama"] = "openai"      # Literal: any other value stops the start
    model: str = ""
    base_url: str = ""
    api_key: str = ""
    # validation_alias: read from OPENAI_API_KEY exactly, with no PORTER_ in front
    openai_api_key: str = Field("", validation_alias="OPENAI_API_KEY")
    temperature: float = 0.0
    model_timeout_s: float = 120.0        # one model call
    model_max_retries: int = 2            # dropped connections, 5xx and 429 only (P5)

    # data: absolute paths, so the process can start from any folder
    db_path: Path = ROOT / "online_retail.db"                # the shop's orders, read-only
    checkpoint_path: Path = ROOT / "porter_threads.sqlite"   # conversations (P4)
    ledger_path: Path = ROOT / "porter_ledger.sqlite"        # return requests and refunds
    postgres_dsn: str = ""                                   # set it and conversations move to Postgres

    # what one request may do (P5)
    max_model_calls: int = 6
    max_tool_calls: int = 8
    request_timeout_s: float = 150.0      # the caller's budget for the whole run

    # who may call it (P3)
    auth_secret: str = "dev-only-not-a-secret-set-PORTER_AUTH_SECRET"    # signs sign-in tokens; .env sets a real one
    token_ttl_s: int = 8 * 3600                                          # a token works for 8 hours

    version: str = "dev"                  # stamped in when the image is built (P7)

    # Runs once every field is loaded: fills in what the provider decides, then checks the fields
    # that only make sense together.
    def model_post_init(self, _context):
        self.model = self.model or PROVIDERS[self.provider]["model"]
        self.base_url = self.base_url or PROVIDERS[self.provider]["base_url"]
        # The OpenAI key goes to OpenAI's address and nowhere else. Any other endpoint gets
        # PORTER_API_KEY if it is set (a tunnel's token), or a placeholder: Ollama checks none.
        if not self.api_key:
            self.api_key = (self.openai_api_key if self.base_url.startswith(PROVIDERS["openai"]["base_url"])
                            else "ollama")
        if self.model_timeout_s >= self.request_timeout_s:
            raise ValueError(f"model_timeout_s = {self.model_timeout_s:.0f}s does not fit "
                             f"inside request_timeout_s = {self.request_timeout_s:.0f}s")

# Read the settings now: from the environment, then .env, then the defaults above.
settings = Settings()
print(settings.provider, "·", settings.model, "at", settings.base_url, "·", settings.db_path.name)

openai · gpt-4.1-mini at https://api.openai.com/v1 · online_retail.db
time: 3.46 ms (started: 2026-09-29 21:45:52 +05:30)


In [17]:
# The ledger: open it (creating its tables the first time), add a return request, count what is in it.
from datetime import datetime, timezone


def open_ledger(path):
    """Open the ledger file, creating it and its two tables the first time."""
    con = sqlite3.connect(path, timeout=10)      # if another connection is writing, wait up to 10 s
    con.row_factory = sqlite3.Row                # rows can be read by column name: row["status"]
    # Two tables:
    #   returns — one row per return request the desk opened; status goes from pending to approved (P6)
    #   refunds — one row per refund a member of staff approved (P6). No tool writes here.
    # IF NOT EXISTS makes this safe to run on every open: it only creates what is missing.
    con.executescript("""
        CREATE TABLE IF NOT EXISTS returns (
            return_id        TEXT PRIMARY KEY,
            idempotency_key  TEXT UNIQUE NOT NULL,     -- the same request twice opens one return
            invoice_no       TEXT NOT NULL,
            customer_id      REAL NOT NULL,
            amount           REAL NOT NULL,
            reason           TEXT,
            status           TEXT NOT NULL DEFAULT 'pending',
            requested_at     TEXT NOT NULL,
            decided_by       TEXT,
            decided_at       TEXT
        );
        CREATE TABLE IF NOT EXISTS refunds (
            refund_id    TEXT PRIMARY KEY,
            return_id    TEXT UNIQUE NOT NULL,         -- the same approval twice pays once
            invoice_no   TEXT UNIQUE NOT NULL,         -- an order is refunded at most once
            customer_id  REAL NOT NULL,
            amount       REAL NOT NULL,
            approved_by  TEXT NOT NULL,
            paid_at      TEXT NOT NULL
        );
    """)
    return con

def now():
    """The time now, in UTC, as text: 2026-09-28T10:15:00+00:00."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def request_return(path, *, key, invoice_no, customer_id, amount, reason):
    """Open a pending return, or find the one this key already opened. Returns (row, created)."""
    con = open_ledger(path)
    try:
        # Add a new request, with a random id such as R-3F2A9C1B and the status 'pending'.
        con.execute("INSERT INTO returns (return_id, idempotency_key, invoice_no, customer_id, amount, reason, requested_at) "
                    "VALUES (?, ?, ?, ?, ?, ?, ?)",
                    (f"R-{uuid.uuid4().hex[:8].upper()}", key, invoice_no, customer_id, amount, reason[:500], now()))
        con.commit()
        created = True
    except sqlite3.IntegrityError:                     # this key already opened one
        # UNIQUE (idempotency_key) refused the row: this exact request was made before.
        # Nothing new is written; the row the first attempt created is returned below.
        created = False
    # Either way, read back the row that belongs to this key.
    row = con.execute("SELECT * FROM returns WHERE idempotency_key = ?", (key,)).fetchone()
    con.close()
    return dict(row), created

def counts(path):
    """What is in the ledger right now. Read this, not the agent's reply."""
    con = open_ledger(path)
    # Return requests by status, e.g. {'pending': 2}; then how many refunds, and their total.
    by_status = dict(con.execute("SELECT status, COUNT(*) FROM returns GROUP BY status").fetchall())
    paid, refunded = con.execute("SELECT COUNT(*), COALESCE(SUM(amount), 0) FROM refunds").fetchone()
    con.close()
    return {"pending": by_status.get("pending", 0), "approved": by_status.get("approved", 0),
            "refunds": paid, "refunded": round(refunded, 2)}


time: 1.73 ms (started: 2026-09-29 21:45:52 +05:30)


In [18]:
# The three tools, built for one customer: two read the orders, start_return writes a request to the ledger.
def read_only(db_path):
    """A connection SQLite itself refuses to write through."""
    # mode=ro opens the file read-only. A plain sqlite3.connect(path) would not do: a DROP TABLE
    # sent through it would be committed.
    return sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, check_same_thread=False)

# Said in every lookup, so the model has no gap to fill with an invented delivery date.
NO_TRACKING = "This desk has no shipping, dispatch or delivery information for any order."

def build_tools(customer_id, db_path, ledger_path, idempotency_key=None, deadline=None):
    """The three tools, bound to one customer and one request."""
    # Everything the tools need is an argument of build_tools(), passed in by the code handling
    # the request. The tools below use these variables directly: they are closures.

    @tool
    def look_up_order(invoice_no: str) -> str:
        """Look up one of this customer's orders: when it was placed, how many lines it has,
        and whether it was cancelled. It cannot say where an order is — there is no shipping
        or delivery information. Use the order number exactly as the customer wrote it."""
        con = read_only(db_path)
        # One row per order: its number, when it was placed, whether it was cancelled, and how many
        # lines it has. "AND i.customer_id = ?" means only this customer's orders are found.
        row = con.execute("""
            SELECT i.invoice_no, i.invoice_ts, i.is_cancelled, COUNT(li.stock_code)
            FROM invoices i JOIN line_items li ON li.invoice_no = i.invoice_no
            WHERE i.invoice_no = ? AND i.customer_id = ?
            GROUP BY i.invoice_no""", (invoice_no.strip(), customer_id)).fetchone()
        con.close()
        if row is None:
            # Not found and not theirs get the same reply.
            return f"No order {invoice_no!r} on this account."
        no, placed, cancelled, lines = row
        return f"Order {no}: placed on {placed}, {lines} line(s), {'cancelled' if cancelled else 'not cancelled'}. {NO_TRACKING}"

    @tool
    def price_order(invoice_no: str) -> str:
        """Itemise one of this customer's orders and give its total. Use this for any question
        about what something cost — never work the total out yourself."""
        con = read_only(db_path)
        # Is the order this customer's?
        owned = con.execute("SELECT 1 FROM invoices WHERE invoice_no = ? AND customer_id = ?",
                            (invoice_no.strip(), customer_id)).fetchone()
        # Its lines with product names, most expensive first. LEFT JOIN keeps a line whose
        # product has no name.
        rows = con.execute("""
            SELECT p.description, li.quantity, li.unit_price
            FROM line_items li LEFT JOIN products p ON p.stock_code = li.stock_code
            WHERE li.invoice_no = ? ORDER BY li.quantity * li.unit_price DESC""",
            (invoice_no.strip(),)).fetchall()
        con.close()
        if owned is None:
            return f"No order {invoice_no!r} on this account."
        # The total is added up here, in Python, so the model never does arithmetic.
        total = round(sum(quantity * price for _, quantity, price in rows), 2)
        # At most twelve lines; a longer order ends with "... and N more line(s)".
        items = [f"  {q} x {(d or 'unknown item').title()} @ {p:.2f}" for d, q, p in rows[:12]]
        if len(rows) > 12:
            items.append(f"  ... and {len(rows) - 12} more line(s)")
        return f"Order {invoice_no} — {len(rows)} line(s), total {total:.2f}\n" + "\n".join(items)

    @tool
    def start_return(invoice_no: str, reason: str) -> str:
        """Open a return request for one of this customer's orders. Nothing is refunded until a
        member of staff approves it. The amount comes from the order, never from the customer's
        message. Call this only once the customer has clearly asked to send something back."""
        # 1. If the caller has already given up, don't start a write nobody will hear about.
        if deadline is not None and time.monotonic() > deadline:      # the caller has given up (P5)
            return "Out of time for this request, so no return was opened."
        # 2. The amount is the order's total, read from the database: never a number the customer typed.
        con = read_only(db_path)
        amount, cancelled = con.execute("""
            SELECT ROUND(SUM(li.quantity * li.unit_price), 2), i.is_cancelled
            FROM invoices i JOIN line_items li ON li.invoice_no = i.invoice_no
            WHERE i.invoice_no = ? AND i.customer_id = ?""", (invoice_no.strip(), customer_id)).fetchone()
        con.close()
        if amount is None:
            return f"No order {invoice_no!r} on this account, so there is nothing to return."
        if cancelled:
            return f"Order {invoice_no} was already cancelled; nothing to return."
        # 3. Record the request in the ledger. Without a key from the caller, every call gets a new
        #    random one, so a repeated request opens a second return. P6 shows why that matters.
        request, created = request_return(ledger_path, key=idempotency_key or f"auto:{uuid.uuid4()}",
                                          invoice_no=invoice_no.strip(), customer_id=customer_id,
                                          amount=amount, reason=reason)
        if not created:                                                # the same request again (P6)
            return (f"Return request {request['return_id']} is already open on order {invoice_no}; "
                    "this was a repeat, so nothing new was opened.")
        return (f"Return request {request['return_id']} opened on order {invoice_no} for {amount:.2f}. "
                "A colleague has to approve it before anything is refunded.")

    return [look_up_order, price_order, start_return]

time: 1.49 ms (started: 2026-09-29 21:45:52 +05:30)


In [19]:
# The tools for customer 12381, called directly: no model involved.
tools = {t.name: t for t in build_tools(CUSTOMER, settings.db_path, settings.ledger_path)}
print("their own order :", tools["look_up_order"].invoke({"invoice_no": "580638"}))
# 536365 is a real order (the first row of invoices above), but it belongs to customer 17850.
print("somebody else's :", tools["look_up_order"].invoke({"invoice_no": "536365"}))

their own order : Order 580638: placed on 2011-12-05 12:44:00, 7 line(s), not cancelled. This desk has no shipping, dispatch or delivery information for any order.
somebody else's : No order '536365' on this account.
time: 7.15 ms (started: 2026-09-29 21:46:11 +05:30)


In [20]:
print("their own order :", tools["price_order"].invoke({"invoice_no": "580638"}))

their own order : Order 580638 — 7 line(s), total 147.01
  1 x Postage @ 40.00
  6 x Set Of 3 Cake Tins Pantry Design  @ 4.95
  12 x Rabbit Night Light @ 2.08
  24 x Pack Of 60 Mushroom Cake Cases @ 0.55
  24 x Pack Of 60 Spaceboy Cake Cases @ 0.55
  24 x Pack Of 72 Retrospot Cake Cases @ 0.55
  1 x Regency Cakestand 3 Tier @ 12.75
time: 3.88 ms (started: 2026-09-29 21:47:30 +05:30)


In [21]:
# The agent for one request: the model, the tools, the system prompt, three limits and a retry rule.
import openai          # the model client's error types; _transient() sorts them into retry and don't retry
from langchain.agents.middleware import ModelCallLimitMiddleware, ModelRetryMiddleware, ToolCallLimitMiddleware

# The model's instructions, sent at the start of every conversation.
SYSTEM_PROMPT = """You are Porter, the order desk for Wickmere & Rook, an online gift shop.

You are talking to a customer about their own orders. Be brief and plain — two or three
sentences, no lists unless you are itemising an order.

How to work:
- Any question about an order goes through a tool. Never state a date, a country, a price
  or a total that a tool did not just give you.
- `look_up_order` for where an order is and what happened to it.
- `price_order` for anything about money. Do not add prices up yourself.
- `start_return` only when the customer has clearly said they want to send something back.
  The refund amount comes from the order, never from what the customer says it was.
- If a tool says there is no such order on this account, say exactly that. Do not guess at
  a different order number.
- If you are asked something that is not about this customer's orders, say it is not
  something the order desk can help with."""

def build_model(settings):
    """The model the settings name: the same client for OpenAI and Ollama, since both serve Chat Completions."""
    return ChatOpenAI(model=settings.model, base_url=settings.base_url, api_key=settings.api_key,
                      temperature=settings.temperature,
                      timeout=settings.model_timeout_s,     # one call's budget (P5)
                      max_retries=0)                         # retrying is the middleware's job (P5)

def _transient(exc):
    """Worth another attempt: a dropped connection, a 5xx, a 429. A timeout is not (P5)."""
    # In the openai package a timeout is a kind of connection error, so it is ruled out first.
    if isinstance(exc, openai.APITimeoutError):
        return False
    return isinstance(exc, (openai.APIConnectionError, openai.InternalServerError, openai.RateLimitError))

def build_agent(customer_id, settings, checkpointer, idempotency_key=None, deadline=None):
    """One request's agent: the model, this customer's tools, the rules, three limits, and the store."""
    return create_agent(
        model=build_model(settings),
        tools=build_tools(customer_id, settings.db_path, settings.ledger_path, idempotency_key, deadline),
        system_prompt=SYSTEM_PROMPT,
        middleware=[                                                        # the ceilings (P5)
            # At most 6 model calls. At the limit the run ends, keeping what it has so far.
            ModelCallLimitMiddleware(run_limit=settings.max_model_calls, exit_behavior="end"),
            # At most 8 tool calls. Past that, each tool call is refused and the model is told why.
            ToolCallLimitMiddleware(run_limit=settings.max_tool_calls, exit_behavior="continue"),
            # A failed model call is tried again twice (after 1 s, then 2 s) if _transient() says
            # it's worth it. If every attempt fails, the error is raised for the route to report.
            ModelRetryMiddleware(max_retries=settings.model_max_retries, retry_on=_transient,
                                 on_failure="error", initial_delay=1.0, backoff_factor=2.0),
        ],
        checkpointer=checkpointer,                                          # where the conversation is kept (P4)
    )

time: 1.17 ms (started: 2026-09-29 21:49:45 +05:30)


In [22]:
# answer(): run the agent on one question and return a typed Answer: the reply, the tools used, the tokens.
from langchain_core.callbacks import get_usage_metadata_callback
from pydantic import BaseModel, Field

# What one answer cost.
class Usage(BaseModel):
    input_tokens: int = 0
    output_tokens: int = 0
    model_calls: int = 0
    tool_calls: int = 0
    seconds: float = 0.0

# What answer() returns, and, from P2, what the web route sends back as JSON.
class Answer(BaseModel):
    thread_id: str
    reply: str
    tools_used: list[str] = []
    usage: Usage = Usage()
    stopped_at_limit: bool = False           # a ceiling ended the run (P5)

def answer(question, customer_id, thread_id=None, *, settings=None, checkpointer=None,
           idempotency_key=None, deadline=None):
    """Answer one question, blocking. Everything it needs is an argument or a setting."""
    settings = settings or Settings()
    thread_id = thread_id or f"t-{uuid.uuid4().hex[:12]}"      # no thread given: a new conversation
    agent = build_agent(customer_id, settings, checkpointer or InMemorySaver(), idempotency_key, deadline)
    t0 = time.perf_counter()
    # Run the agent to the end. The callback adds up the tokens of every model call inside it.
    with get_usage_metadata_callback() as usage:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]},
                              {"configurable": {"thread_id": thread_id}})
    return _collect(result, thread_id, usage.usage_metadata, time.perf_counter() - t0)       


# What the customer reads when a limit ended the run (P5), instead of a half-finished reply.
LIMIT_REPLY = "That needs more steps than I can take in one go. Could you ask about one order at a time?"

def _collect(result, thread_id, usage, seconds):
    """The graph's final state, as an Answer. Only this turn's messages count."""
    messages = result["messages"]            # the whole conversation so far, oldest first
    # This turn starts at the last message the customer sent; everything before it is earlier turns.
    last_question = max(i for i, m in enumerate(messages) if m.type == "human")
    reply, tools_used, model_calls, stopped = "", [], 0, False
    for m in messages[last_question + 1:]:
        if m.type != "ai":                    # a tool's result, not a model call
            continue
        if not m.usage_metadata:              # a limit's own message: no model wrote it (P5)
            stopped = True
            continue
        model_calls += 1
        tools_used += [call["name"] for call in m.tool_calls]     # the tools this call asked for
        reply = m.content or reply            # the reply is the last model message with text in it
    return Answer(
        thread_id=thread_id,
        reply=(LIMIT_REPLY if stopped else reply).strip(),
        tools_used=tools_used,
        usage=Usage(input_tokens=sum(u.get("input_tokens", 0) for u in usage.values()),
                    output_tokens=sum(u.get("output_tokens", 0) for u in usage.values()),
                    model_calls=model_calls, tool_calls=len(tools_used), seconds=round(seconds, 3)),
        stopped_at_limit=stopped,
    )           

time: 4.47 ms (started: 2026-09-29 21:53:40 +05:30)


In [23]:
memory = InMemorySaver()

a1 = answer(
    "How much was order 580638?",
    customer_id=12381,
    thread_id="conversation-1",
    checkpointer=memory,
)

print("For q1: ", a1.reply, a1.usage.model_dump())

a2 = answer(
    "And when was that one placed?",
    customer_id=12381,
    thread_id="conversation-1",
    checkpointer=memory,
)

print("For q2: ", a2.reply, a2.usage.model_dump())

For q1:  Order 580638 cost a total of 147.01. It included postage and various cake tins, cake cases, rabbit night lights, and a cakestand. {'input_tokens': 1030, 'output_tokens': 50, 'model_calls': 2, 'tool_calls': 1, 'seconds': 3.508}
For q2:  Order 580638 was placed on December 5, 2011. It has 7 lines and was not cancelled. {'input_tokens': 1349, 'output_tokens': 43, 'model_calls': 2, 'tool_calls': 1, 'seconds': 2.196}
time: 5.76 s (started: 2026-09-29 21:53:54 +05:30)


# Part 2

```
Server starts
    ↓
lifespan() initializes shared resources
    ↓
HTTP request arrives
    ↓
request_id middleware assigns request ID
    ↓
FastAPI validates request
    ↓
route executes
    ↓
if error → appropriate exception handler
    ↓
middleware adds response headers
    ↓
response returned
```

In [24]:
class ChatRequest(BaseModel):
    question: str = Field(min_length=1, max_length=2000)
    thread_id: str | None = None             # continue one of your conversations; omit to start one
    customer_id: float | None = None         # who is asking; this first version believes it (P3)



time: 849 µs (started: 2026-09-29 21:58:02 +05:30)


In [ ]:
# The FastAPI app: start-up (lifespan), a request id on every response, one error shape for every failure.
from contextlib import asynccontextmanager
from fastapi import Depends, FastAPI, Header, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer



time: 207 ms (started: 2026-09-29 21:58:18 +05:30)


In [27]:
class ApiError(Exception):
    """A failure with a code a client can branch on."""
    def __init__(self, status, code, detail):
        super().__init__(detail)
        self.status, self.code, self.detail = status, code, detail

#raise ApiError(
#    404,
#    "conversation_not_found",
#    "That conversation does not exist."
#)

# Where the app keeps conversations. The code before `yield` opens the store and the code after
# it closes it (there's nothing to close for memory). P4 writes this function again.
@asynccontextmanager
async def open_async_checkpointer(settings):
    """Where the app keeps conversations. First version: the process's own memory."""
    yield InMemorySaver()


# e.g.
# with open("file.txt") as f:
#     ...

# startup
#     ↓
# application runs
#     ↓
# shutdown


# Runs up to `yield` when the app starts; the app serves requests while it waits there; the rest
# runs when the app stops.
@asynccontextmanager
async def lifespan(app):
    settings = Settings()
    open_ledger(settings.ledger_path).close()       # a missing ledger volume fails the start, not a customer
    async with open_async_checkpointer(settings) as saver:
        # app.state holds what every request needs; a route reaches it as request.app.state.
        app.state.settings, app.state.checkpointer = settings, saver
        yield

app = FastAPI(title="Porter — Wickmere & Rook order desk", lifespan=lifespan)


bearer = HTTPBearer(auto_error=False)

# HTTP middleware: runs around every request. call_next(request) runs the route and returns its response.

#                     ┌─────────────┐
# request ───────────▶│ middleware  │
#                     │             │
#                     │   route     │
#                     │             │
# response ◀──────────│ middleware  │
#                     └─────────────┘
@app.middleware("http")
async def request_id(request: Request, call_next):
    """One id per request: in every error body, and on the response."""
    request.state.request_id = uuid.uuid4().hex[:12]
    response = await call_next(request)
    response.headers["x-request-id"] = request.state.request_id
    response.headers["x-porter-version"] = request.app.state.settings.version     # which version answered
    return response

# Every failure leaves the app in this one shape: {error, detail, request_id}.
def error(request, status, code, detail):
    body = {"error": code, "detail": detail, "request_id": getattr(request.state, "request_id", "-")}
    return JSONResponse(status_code=status, content=body)

# The request didn't match ChatRequest: report the first problem, and which field it is in.
@app.exception_handler(RequestValidationError)
async def on_invalid_request(request, exc):
    first = exc.errors()[0]
    where = ".".join(str(part) for part in first["loc"][1:]) or "body"     # ("body", "question") → "question"
    return error(request, 422, "invalid_request", f"{where}: {first['msg']}")

# A route raised ApiError: send its status, code and detail.
@app.exception_handler(ApiError)
async def on_api_error(request, exc):
    return error(request, exc.status, exc.code, exc.detail)

# Anything else: a 500 that says nothing about the inside.
@app.exception_handler(Exception)
async def on_unhandled(request, exc):
    """The traceback carries the database path, the SQL and the prompt, so it stays out of the reply."""
    return error(request, 500, "internal", "Something went wrong on our side.")

time: 1.51 ms (started: 2026-09-29 22:01:34 +05:30)


In [28]:
# The health check, and the GET /healthz route that reports it.
async def check_health(settings):
    """Each dependency, checked for real and reported separately: 'ok', or what went wrong."""
    checks = {}
    # 1. The orders: run one real query, rather than asking whether the file exists.
    try:
        con = read_only(settings.db_path)
        con.execute("SELECT 1 FROM invoices LIMIT 1")
        con.close()
        checks["database"] = "ok"
    except Exception as exc:
        checks["database"] = f"failed: {type(exc).__name__}: {exc}"

    # 2. The model: GET /models lists the models the endpoint serves. Is ours one of them?
    #    Only OpenAI is ever left without a key (Settings); sent empty, it isn't even valid HTTP.
    try:
        if not settings.api_key:
            checks["model"] = "no API key: set OPENAI_API_KEY"
        else:
            async with httpx.AsyncClient(timeout=3) as http:
                r = await http.get(f"{settings.base_url}/models", headers={"Authorization": f"Bearer {settings.api_key}"})
            if r.status_code != 200:                            # 401: the key was refused
                checks["model"] = f"http {r.status_code} from {settings.base_url}"
            else:
                served = {m["id"] for m in r.json().get("data", [])}
                checks["model"] = "ok" if settings.model in served else f"{settings.model} is not served there"
    except Exception as exc:
        checks["model"] = f"unreachable: {type(exc).__name__} at {settings.base_url}"

    # 3. The ledger: can it be opened (or created)?
    try:
        open_ledger(settings.ledger_path).close()
        checks["ledger"] = "ok"
    except Exception as exc:
        checks["ledger"] = f"failed: {type(exc).__name__}: {exc}"
    return checks

# GET /healthz: "ok" only if every check is ok; otherwise "degraded", with the failing check named.
@app.get("/healthz")
async def healthz(request: Request):
    settings = request.app.state.settings
    checks = await check_health(settings)
    return {"status": "ok" if all(v == "ok" for v in checks.values()) else "degraded",
            "version": settings.version, "model": settings.model, "checks": checks}

time: 1.69 ms (started: 2026-09-29 22:04:10 +05:30)


In [29]:
# Who is calling, as every route sees it once identify() has decided.
class Identity(BaseModel):
    subject: str                    # "customer:12381" or "staff:alice"
    role: str                       # "customer" or "staff"
    customer_id: float | None = None
    namespace: str                  # the prefix this caller's conversations are stored under

def identify(request, credentials, claimed_customer):
    """First version: believe whichever customer the request names. P3 writes this function again."""
    if claimed_customer is None:
        raise ApiError(422, "invalid_request", "customer_id is required.")
    return Identity(subject=f"customer:{int(claimed_customer)}", role="customer",
                    customer_id=claimed_customer, namespace=f"c{int(claimed_customer)}")

async def find_thread(request, namespace, thread_id):
    """(public id, storage key). A new conversation gets a new id."""
    if thread_id:
        key = f"{namespace}:{thread_id}"       # "c12381:t-…": the same public id under another namespace is another key
        # aget_tuple returns the saved conversation, or None if nothing is stored under that key.
        if await request.app.state.checkpointer.aget_tuple({"configurable": {"thread_id": key}}) is None:
            raise ApiError(404, "thread_not_found", f"No conversation {thread_id!r} on this account.")
        return thread_id, key
    public = f"t-{uuid.uuid4().hex[:12]}"      # no thread_id: start a new conversation
    return public, f"{namespace}:{public}"

time: 1.63 ms (started: 2026-09-29 22:08:20 +05:30)


In [30]:
# aanswer() and two routes: POST /v1/chat answers a question, GET /v1/threads/{id} shows a conversation.
async def aanswer(question, customer_id, thread_id=None, *, settings=None, checkpointer=None,
                  idempotency_key=None, deadline=None):
    """The same answer, awaited, so a caller that runs out of time can cancel it (P5)."""
    # Line for line answer() from P1, with ainvoke in place of invoke.
    settings = settings or Settings()
    thread_id = thread_id or f"t-{uuid.uuid4().hex[:12]}"
    agent = build_agent(customer_id, settings, checkpointer or InMemorySaver(), idempotency_key, deadline)
    t0 = time.perf_counter()
    with get_usage_metadata_callback() as usage:
        result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]},
                                     {"configurable": {"thread_id": thread_id}})
    return _collect(result, thread_id, usage.usage_metadata, time.perf_counter() - t0)

# POST /v1/chat. response_model=Answer: what the route returns is checked against Answer and sent as JSON.
@app.post("/v1/chat", response_model=Answer)
async def chat(body: ChatRequest, request: Request,
               credentials: HTTPAuthorizationCredentials | None = Depends(bearer),
               idempotency_key: str | None = Header(default=None, alias="Idempotency-Key")):
    settings = request.app.state.settings
    identity = identify(request, credentials, body.customer_id)                      # who is asking
    public, key = await find_thread(request, identity.namespace, body.thread_id)     # which conversation
    deadline = time.monotonic() + settings.request_timeout_s                         # when the caller gives up
    try:
        # Run the agent, waiting at most request_timeout_s. Past that, wait_for cancels the run (P5).
        result = await asyncio.wait_for(
            aanswer(body.question, identity.customer_id, key, settings=settings,
                    checkpointer=request.app.state.checkpointer,
                    idempotency_key=idempotency_key, deadline=deadline),
            timeout=settings.request_timeout_s)
    except (TimeoutError, openai.APITimeoutError):          # by type, never by message
        raise ApiError(504, "timeout", "The desk took too long to answer. Please try again.")
    except (openai.APIConnectionError, openai.RateLimitError):   # unreachable, or rate-limited / out of quota
        raise ApiError(503, "model_unavailable", "The desk is temporarily unavailable.")
    # Inside, the thread is "c12381:t-…"; the caller only ever sees the public "t-…".
    return result.model_copy(update={"thread_id": public})

# GET /v1/threads/{id}: one conversation's messages. customer_id comes from the query string (?customer_id=…).
@app.get("/v1/threads/{thread_id}")
async def get_thread(thread_id: str, request: Request, customer_id: float | None = None,
                     credentials: HTTPAuthorizationCredentials | None = Depends(bearer)):
    identity = identify(request, credentials, customer_id)
    # Looked up only under the caller's own namespace.
    found = await request.app.state.checkpointer.aget_tuple(
        {"configurable": {"thread_id": f"{identity.namespace}:{thread_id}"}})
    if found is None:
        raise ApiError(404, "thread_not_found", f"No conversation {thread_id!r} on this account.")
    messages = found.checkpoint["channel_values"]["messages"]          # the saved conversation
    # Every message's role, and its first 300 characters.
    return {"thread_id": thread_id, "turns": len(messages),
            "messages": [{"role": m.type, "text": m.content[:300]} for m in messages]}

time: 3.8 ms (started: 2026-09-29 22:22:11 +05:30)


In [31]:
# Start the app inside this process, and write a helper that sends it a question.
from fastapi.testclient import TestClient

client = TestClient(app).__enter__()          # runs lifespan's startup, as a server would

def ask(question, customer_id=CUSTOMER, **fields):
    """One question to the desk, over HTTP, in this process. The body says who is asking."""
    return client.post("/v1/chat", json={"question": question, "customer_id": customer_id, **fields})

# The health check first: all three dependencies should say ok.
print(json.dumps(client.get("/healthz").json(), indent=2))

{
  "status": "ok",
  "version": "dev",
  "model": "gpt-4.1-mini",
  "checks": {
    "database": "ok",
    "model": "ok",
    "ledger": "ok"
  }
}
time: 1.44 s (started: 2026-09-29 22:22:38 +05:30)


In [ ]:
# Making sign-in tokens, and checking them.
import jwt           # PyJWT: makes and checks JSON Web Tokens

def issue_token(customer_id=None, staff=None, secret=None):
    """Sign a token. In production the shop's login system does this, not the desk."""
    settings = Settings()
    # The claims: who this is (sub) and what kind of caller (role).
    claims = {"sub": f"staff:{staff}", "role": "staff"} if staff else \
             {"sub": f"customer:{int(customer_id)}", "role": "customer"}
    issued = int(time.time())
    claims.update(iat=issued, exp=issued + settings.token_ttl_s)     # issued at, and expires at, in seconds
    # HS256 signs the claims with the secret. Change one character of the token and the signature no longer matches.
    return jwt.encode(claims, secret or settings.auth_secret, algorithm="HS256")

def decode_token(token, secret):
    """Check the signature and the expiry, then believe the claims."""
    # jwt.decode raises if the signature is wrong, the token has expired, or exp or sub is missing.
    claims = jwt.decode(token, secret, algorithms=["HS256"], options={"require": ["exp", "sub"]})
    if claims.get("role") == "staff":
        return Identity(subject=claims["sub"], role="staff", namespace=claims["sub"].replace(":", "-"))
    customer_id = float(claims["sub"].partition(":")[2])          # "customer:12381" → 12381.0
    return Identity(subject=claims["sub"], role="customer", customer_id=customer_id,
                    namespace=f"c{int(customer_id)}")

# A token is three dot-separated parts: header.claims.signature. The first two are only base64, so anyone can read them.
print(issue_token(customer_id=CUSTOMER)[:60], "…")